# XRF55 Bench — WavDualMamba

**Model:** WavDualMamba — multi-branch (per-DWT-subband) CNN + gated BiMamba, adaptive late fusion.  
**Dataset:** XRF55 — 11 activities, 4620 train / 1980 test  
**Split:** train = reps 1–14, test = reps 15–20. No val.

Keeps the subcarrier axis as features (no FreqAttnPool collapse); fwd/bwd merged by a per-channel zero-init gate; subbands fused by AdaptiveFusion.

---

## Attached dataset

| Dataset name | Mount path |
|---|---|
| `xrf55-amp-dataset` | `/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/` |

## Protocol reference

| # | Optimizer | LR | WD | Scheduler | Epochs | Notes |
|---|---|---|---|---|---|---|
| 01 | Adam | 1e-3 | 0.0 | MultiStepLR [40,80,120,160] γ=0.5 | 200 | XRF55 paper |
| 02 | AdamW | 5e-4 | 1e-3 | warmup(10ep)+cosine → 4e-5 | 200 | APWMamba paper |

## Ablation flags (`model_kwargs`)

### Architecture
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `subbands` | `('LL','HL','LH')` | any subset | Which DWT subbands → branches |
| `share_branches` | `False` | `True` / `False` | Tie backbone weights across subbands (stems stay per-subband) |
| `fusion` | `'convex'` | `'convex'` / `'gate'` / `'concat'` | Branch-merge: `convex`=per-branch scalar (baseline), `gate`=per-channel convex routing, `concat`=static full-matrix mix. All zero-init → mean(streams) at step 0 |

### CNN stage
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `temporal_stride` | `1` | `1` / `2` | Stride in stem conv time axis; `2` → T 500→250 |
| `dilations` | `(1, 2, 4)` | tuple | TFBlock dilation schedule |
| `dp_cnn` | `(0.0, 0.05, 0.1)` | tuple | DropPath per TFBlock |
| `freq_mix` | `None` | `None` / `'mlp'` | Nonlinear subcarrier mix before flatten |

### Embed
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `embed_drop` | `0.1` | float | Dropout after Linear embed |
| `embed_hidden` | `None` | `int` / `None` | Two-stage embed `240→hidden→d_model` |

### Mamba
| Flag | Default | Values | Meaning |
|---|---|---|---|
| `d_model` | `64` | int | Feature width (v2 uses 96 — set here to compare fairly) |
| `d_state` | `32` | int | SSM state size (v2 uses 16) |
| `n_mamba_layers` | `2` | int | Number of BiMamba layers |
| `ffn_ratio` | `0` | `0` / `2` | FFN sub-block after each Mamba layer; `0` = no FFN |
| `bidirectional` | `True` | `True` / `False` | Gated backward Mamba branch |
| `dp_mamba` | `(0.0, 0.10)` | tuple | DropPath per BiMamba layer |
| `expand` | `2` | int | Mamba inner-expansion |
| `d_conv` | `4` | int | Mamba local conv width |
| `use_pos_emb` | `False` | `True` / `False` | Sinusoidal positional embedding |

### C6/C7/C8 (new ablation flags — default ON)

> **Note:** `use_eca=True` and `pool_context=True` are new defaults and are NOT active in old v1 runs. Set both to `False` to reproduce exact old v1 behaviour.

| Flag | Default | Values | Meaning |
|---|---|---|---|
| `use_eca` | `True` | `True` / `False` | [C7] ECA channel gate (~5 params) on raw 27-ch input before stems; `False` = old v1 |
| `pool_context` | `True` | `True` / `False` | [C8] AttnStatPool sees `[x‖μ‖σ]` (full ECAPA context); `False` = old v1 |
| `use_final_attn` | `False` | `True` / `False` | [C6] One MHSA layer after fusion, before pooling; `True` = ablation |
| `attn_heads` | `4` | int | MHSA heads for FinalAttention (only used when `use_final_attn=True`) |

## Param budget (ước tính)

| Config | Params | Note |
|---|---|---|
| baseline (d_model=64, use_eca=F, pool_context=F) | ~582K | old v1 behaviour |
| + use_eca=True | ~582K | ECA adds ~5 params |
| + pool_context=True | ~590K | larger score network (3×dim input) |
| `d_model=96` | ~840K | same width as v2 (unfair compare unless other v2 flags match) |
| `ffn_ratio=2` | ~680K | +FFN sub-block |
| `embed_hidden=128` | ~653K | +two-stage embed |
| `use_final_attn=True` | ~597K | +FinalAttention (~15K at d=64, h=4) |
| `fusion='gate'` | ~+37K | per-channel branch routing |
| `fusion='concat'` | ~+12K | static full-matrix merge |

In [ ]:
# Cell 1 — Install mamba-ssm (required by WavDualMamba BiMamba layers)
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
print('Install done')

In [ ]:
# Cell 2 — Clone / update latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/xrf55_benchmark')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/xrf55_benchmark.git',
         str(CODE_PATH)],
        check=True
    )
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from xrf55_bench.trainer import run
print('Import OK  : xrf55_bench.trainer.run')

In [ ]:
# Cell 3 — Configuration
from pathlib import Path
from xrf55_bench.config import TrainCfg, TrainCfg_for_protocol

SEEDS      = [42]               # single-seed default
# SEEDS    = [0, 4, 8, 17, 42] # multi-seed

# Data source — uncomment one:
# BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/raw')
BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/processed')

# ── Model kwargs — chỉnh trực tiếp, xoá dòng nào không muốn override ─────────
MODEL_KWARGS = {
    # ── Architecture ────────────────────────────────────────────────────────
    'subbands'        : ('LL', 'HL', 'LH'),  # ('HL','LH') | ('LL','HL') | ('LL','LH')
    'share_branches'  : False,               # True → tied weights, batched call
    'fusion'          : 'convex',            # 'gate' = per-kênh routing | 'concat' = ma trận tĩnh | 'convex' = baseline

    # ── CNN stage ───────────────────────────────────────────────────────────
    'temporal_stride' : 1,                   # 2 → T 500→250 (~2× faster)
    'dilations'       : (1, 2, 4),           # (1,2,4) → 3 TFBlocks
    'dp_cnn'          : (0.0, 0.05, 0.1),   # DropPath per TFBlock (len == len(dilations))
    'freq_mix'        : None,                # 'mlp' → nonlinear subcarrier mix

    # ── Embed ───────────────────────────────────────────────────────────────
    'embed_drop'      : 0.1,                 # Dropout sau Linear embed
    'embed_hidden'    : None,                # 128 → two-stage 240→128→d_model

    # ── Mamba ───────────────────────────────────────────────────────────────
    'd_model'         : 64,                  # feature width (v2 default: 96)
    'd_state'         : 32,                  # SSM state size (v2 default: 16)
    'n_mamba_layers'  : 2,                   # BiMamba depth
    'ffn_ratio'       : 0,                   # 2 → FFN sub-block sau mỗi Mamba layer
    'bidirectional'   : True,                # False → unidirectional (ablation)
    'dp_mamba'        : (0.0, 0.10),         # DropPath per BiMamba layer (len == n_mamba_layers)
    'expand'          : 2,                   # Mamba inner-expansion
    'd_conv'          : 4,                   # Mamba local conv width
    'use_pos_emb'     : False,               # True → sinusoidal PE

    # ── C6/C7/C8 ablation flags ─────────────────────────────────────────────
    # NOTE: use_eca=True and pool_context=True are NEW defaults (not in old v1 runs).
    # To reproduce exact old v1 behaviour: set both to False.
    'use_eca'         : False,   # [C7] ECA gate on raw 27-ch input; True = ablation
    'pool_context'    : True,    # [C8] full ECAPA [x‖μ‖σ] pooling; False = old v1
    'use_final_attn'  : False,   # [C6] MHSA after fusion, before pool; True = ablation
    'attn_heads'      : 4,       # heads for FinalAttention (only used if use_final_attn=True)
}

# ── Auto output-dir tag từ ablation config ───────────────────────────────────
_tag = '_'.join(MODEL_KWARGS.get('subbands', ('LL', 'HL', 'LH'))).lower()
if MODEL_KWARGS.get('temporal_stride', 1) == 2:             _tag += '_t250'
if MODEL_KWARGS.get('share_branches'):                       _tag += '_shared'
if MODEL_KWARGS.get('fusion', 'convex') != 'convex':        _tag += f"_{MODEL_KWARGS['fusion']}"
if MODEL_KWARGS.get('freq_mix'):                             _tag += f"_{MODEL_KWARGS['freq_mix']}"
if MODEL_KWARGS.get('use_pos_emb'):                         _tag += '_pe'
if MODEL_KWARGS.get('bidirectional') is False:               _tag += '_uni'
if MODEL_KWARGS.get('ffn_ratio', 0) > 0:                    _tag += f"_ffn{MODEL_KWARGS['ffn_ratio']}"
if MODEL_KWARGS.get('d_model', 64) != 64:                   _tag += f"_d{MODEL_KWARGS['d_model']}"
if MODEL_KWARGS.get('d_state', 32) != 32:                   _tag += f"_s{MODEL_KWARGS['d_state']}"
if MODEL_KWARGS.get('n_mamba_layers', 2) != 2:              _tag += f"_L{MODEL_KWARGS['n_mamba_layers']}"
if MODEL_KWARGS.get('embed_hidden'):                         _tag += f"_emb{MODEL_KWARGS['embed_hidden']}"
if not MODEL_KWARGS.get('use_eca', True):                   _tag += '_noeca'
if not MODEL_KWARGS.get('pool_context', True):              _tag += '_nopool'
if MODEL_KWARGS.get('use_final_attn', False):               _tag += '_attn'
OUTPUT_DIR = Path(f'/kaggle/working/outputs/wavdual_{_tag}_p02_proc')

print(f'Seeds      : {SEEDS}')
print(f'Bench dir  : {BENCH_DIR}')
print(f'Model cfg  : {MODEL_KWARGS}')
print(f'Output dir : {OUTPUT_DIR}')

for fname in ['stats.json', 'y_train.npy', 'y_test.npy',
              'wavmamba/X_train.npy', 'wavmamba/X_test.npy']:
    p = BENCH_DIR / fname
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p}')

In [ ]:
# Cell 4 — Run training
#
# Protocol presets:
#   01  Adam   lr=1e-3  wd=0.0   MultiStepLR     200 ep  (XRF55 paper)
#             milestones=[40,80,120,160]  gamma=0.5
#             milestones=[40,80,120,160]  gamma=0.5
#   02  AdamW  lr=5e-4  wd=1e-3  warmup+cosine   200 ep  (APWMamba paper)
#             warmup=10ep  floor_lr=4e-5  label_smoothing=0.0
#
# LR overrides (None = keep the protocol default):
MAX_LR   = 5e-4     # peak LR / warmup target, e.g. 5e-4
FLOOR_LR = 1e-6     # final LR floor for cosine / warmup_cosine, e.g. 4e-5
_lr_over = {k: v for k, v in (('lr', MAX_LR), ('floor_lr', FLOOR_LR)) if v is not None}

# Uncomment one cfg line:
# cfg = TrainCfg_for_protocol('01', seeds=tuple(SEEDS), **_lr_over)          # 200ep, Adam  lr=1e-3, MultiStepLR
# cfg = TrainCfg_for_protocol('02', seeds=tuple(SEEDS), **_lr_over)          # 200ep, AdamW lr=5e-4, warmup+cosine
cfg = TrainCfg_for_protocol('02', seeds=tuple(SEEDS), num_epochs=80, betas=(0.9, 0.95), grad_clip=1.0, **_lr_over)  # 80ep, AdamW lr=5e-4 → 1e-6, gc=1

run(model_name='wavdualmamba', bench_dir=BENCH_DIR, output_dir=OUTPUT_DIR,
    train_cfg=cfg, num_workers=4, model_kwargs=MODEL_KWARGS)

In [ ]:
# Cell 4b (optional) — Sweep all 4 subband ablations in one run
# Each ablation writes to its own auto-tagged OUTPUT_DIR.
#
# SUBBAND_STUDIES = [('HL', 'LH'), ('LL', 'HL'), ('LL', 'LH'), ('LL', 'HL', 'LH')]
# for sb in SUBBAND_STUDIES:
#     tag = '_'.join(sb).lower()
#     out = Path(f'/kaggle/working/outputs/wavdual_{tag}_p02_raw')
#     cfg = TrainCfg_for_protocol('02', seeds=tuple(SEEDS))
#     print(f'\n########## subbands={sb} -> {out} ##########')
#     run(model_name='wavdualmamba', bench_dir=BENCH_DIR, output_dir=out,
#         train_cfg=cfg, num_workers=4, model_kwargs={'subbands': sb})

In [ ]:
# Cell 5 — Results
import json

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    s     = m['summary']
    seeds = m['config']['seeds']
    print(f"\n{'='*55}")
    print(f"  XRF55 Bench — WavDualMamba")
    print(f"  Model cfg   : {m.get('model_config', {})}")
    print(f"  Seeds       : {seeds}")
    print(f"  Accuracy    : {s['test_accuracy_mean']*100:.2f}% ± {s['test_accuracy_std']*100:.2f}%")
    print(f"  F1 Macro    : {s['test_f1_macro_mean']*100:.2f}% ± {s['test_f1_macro_std']*100:.2f}%")
    print(f"  Best epochs : {s['best_epochs']}")
    print(f"  Params      : {s['params_M']}M  |  Size: {s['model_size_mb']}MB")
    if s.get('macs_G'):
        print(f"  MACs        : {s['macs_G']:.3f}G")
    if s.get('latency_mean_ms') is not None:
        print(f"  Latency     : {s['latency_mean_ms']:.2f} ± {s['latency_std_ms']:.2f} ms")
    print(f"  Time        : {s['total_time_s']}s")
    print(f"{'='*55}")
    if len(seeds) == 1:
        ps = m['per_seed'].get(str(seeds[0]), {})
        if ps.get('test_f1_per_class'):
            class_names = [
                'Waving', 'Clap Hands', 'Fall on the Floor', 'Jumping', 'Running',
                'Sitting Down', 'Standing Up', 'Turning', 'Walking',
                'Stretch Oneself', 'Pat on Shoulder',
            ]
            print(f"\n  Per-class F1:")
            for i, v in enumerate(ps['test_f1_per_class']):
                print(f"    {class_names[i]:<20}: {v*100:.2f}%")
else:
    print('metrics.json not found — training may not have completed.')

In [ ]:
# Cell 6 — Plots + Download
import shutil
from IPython.display import Image, display, FileLink

for fname in ['training_curve.png', 'confusion_matrix.png', 'seed_comparison.png']:
    p = OUTPUT_DIR / 'plots' / fname
    if p.exists():
        display(Image(str(p)))

print()
print('--- Download ---')
zips = sorted(OUTPUT_DIR.glob('*.zip'))
if zips:
    for src in zips:
        dst = Path('/kaggle/working') / src.name
        shutil.copy2(src, dst)
        size_mb = dst.stat().st_size / 1e6
        print(f'{src.name}  ({size_mb:.1f} MB)')
        display(FileLink(src.name))
else:
    print('[MISSING] no zip found — run Cell 4 first.')